<a href="https://colab.research.google.com/github/elhamod/IS883_Fall_2026/blob/main/Session%2003/IS883_Session03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deploying an LLM Web App: Calling the Gemini API


# Part 0: Setup

In [ ]:
# Enter your BUID. This week it is handed to the Gemini API as its `seed`, which is
# what makes a call repeatable -- there is no local model left for us to seed.
BUID = 123456  # enter ONLY the numerical part


## Part 1: Using the Gemini API

In [ ]:
# Install Google's GenAI SDK for the Gemini API.
!pip install -U google-genai

### Generate Text with the Gemini API.

Now that you have already experimented with loading a language model (GPT2) *locally* and using it to generate some sentences last week, how about we instead use someone else's model through an API? Let's experiment with Google's Gemini API!

Unlike most commercial APIs, Gemini has a **free tier** that does not require a credit card, which is why we are using it in this course.

- In order to use the Gemini API, you first need to get an API key. You can create one for free through [this link](https://aistudio.google.com/apikey) after signing in with a Google account.
- Once you have created the key, you will save it as a secret in Google Colab (the key icon 🔑 in the left sidebar → *Add new secret* → make sure **Notebook access** is switched on). For grading purposes, you MUST name your key *MyGeminiKey*.
- **Never paste the key directly into a code cell.** Anyone you share the notebook with would then be able to spend your quota.
- Now, you are set! Use the [Gemini API documentation](https://ai.google.dev/gemini-api/docs/text-generation) to complete the same two prefixes from Week 2. **(10 Points)**
  - You will use the *gemini-3.1-flash-lite* model.
  - You will generate up to 20 tokens per request.
  - You will generate 5 different completions.
  - You will set the seed based on your BUID.
  - Make sure the API call *completes* the given prefix (i.e., it does not start a new sentence).

In [ ]:
# Load an API key. In Colab it comes from the Secrets panel; anywhere else
# (VS Code, a local Jupyter) it comes from an environment variable, and failing
# both you are asked to paste it. The key is never written into this notebook.
import os, getpass

def get_key(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name) or getpass.getpass(f"{name}: ")


In [ ]:
# Connect to the Gemini API using the key you saved as a Colab secret, and check it works.
from google import genai
from google.genai import types

client = genai.Client(api_key=get_key('MyGeminiKey'))

MODEL = "gemini-3.1-flash-lite"

# Quick check that the key works
print(client.models.generate_content(model=MODEL, contents="Say hello in five words.").text)

model_info = client.models.get(model=MODEL)
print(f"Input token limit: {model_info.input_token_limit}")
print(f"Output token limit: {model_info.output_token_limit}")

In [ ]:
# Helper: send ONE request to Gemini, print how many tokens it used, and return the text.
import time

def ask_gemini(prompt, system_instruction=None, seed=None, temperature=None,
               max_output_tokens=None, stop_sequences=None):
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        seed=seed,
        temperature=temperature,
        max_output_tokens=max_output_tokens,
        stop_sequences=stop_sequences,
        thinking_config=types.ThinkingConfig(
            thinking_level="MINIMAL",   # closest thing to "off" on Gemini 3
            include_thoughts=False,     # don't return thought summaries
        ),
        tools=[],                       # no tools
    )

    # Retry a few times if we hit the free tier's rate limit.
    for attempt in range(4):
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=config,
            )

            # Show how many tokens this request used (this is what an API bills for).
            usage = response.usage_metadata
            print("*" * 20)
            print("input_tokens:   ", usage.prompt_token_count)
            print("output_tokens:  ", usage.candidates_token_count)
            print("thinking_tokens:", usage.thoughts_token_count)
            print("tools_tokens:   ", usage.tool_use_prompt_token_count)
            print("cached_tokens:  ", usage.cached_content_token_count)
            print("total_tokens:   ", usage.total_token_count)
            print("*" * 20)

            return response.text or "[the model returned no text]"
        except Exception as error:
            if "429" in str(error) or "RESOURCE_EXHAUSTED" in str(error):
                print("   (free-tier rate limit reached, waiting 15 seconds...)")
                time.sleep(15)
            else:
                raise

    return "[gave up after repeated rate limits]"

In [ ]:
# Ask Gemini to complete each prefix five times (a fixed seed per run makes it repeatable).
NUM_COMPLETIONS = 5

prompts = ["Damascus is a", "Barcelona is a"]

for prompt in prompts:
    for i in range(NUM_COMPLETIONS):
        completion = ask_gemini(
            prompt=prompt,
            system_instruction="Complete the following prefix. Do not start a new sentence.",
            seed=BUID + i,
            max_output_tokens=20,
        )
        print(f"{i + 1}. {prompt} ... {completion}")

**Experiment.** Try changing the parts of the call above and re-running:
- **`prompts`** — swap in other prefixes of your own.
- **`system_instruction`** — this is the model's standing order. Why does `"Complete the following prefix. Do not start a new sentence."` matter here, and what happens if you remove or change it?
- **`max_output_tokens`** — this caps how long each completion can be. What changes if you raise or lower it?

## Follow up Questions and Exercises — Part 1: the Gemini API

**Question 1.** What do you notice about the generated texts for the two prompts? Any interesting commonalities or stark differences? *(5 points)*

**Answer**

*Provide your answer here*

**Question 2.** How do the completions above compare to those from Week 2 (using GPT2)? What do you think is the underlying cause? *(5 points)*

**Answer**

*Provide your answer here*

# Part 2: Parameter Tweaking

## 2.1 Getting a Structured Answer Instead of Prose

Every response so far has come back as a paragraph of English. That is fine when a human is reading it, and useless when a *program* is. If you want to process 500
customer reviews, you do not want 500 paragraphs — you want 500 rows.

Below we ask the exact same question twice, and try to pull data out of both answers
using the exact same extraction code. The only difference is one line of
configuration: `response_schema`.

Case 1 will fail. Case 2 will not.

Notice the last field in the form. "Poorest" is not one thing: GDP per capita,
PPP-adjusted income, and multidimensional poverty indices point to *different
countries*. Forcing the model to state which measure it used turns a hidden
assumption into a visible column.

In [ ]:
# Show why response_schema matters: the same question, with and without a required output shape.
import json
from pydantic import BaseModel

QUESTION = ("Which country is the poorest in the world? Give its capital, GDP per capita, and population.")


# The "form" we want filled in. Every field is required.
class CountryProfile(BaseModel):
    country: str
    capital: str
    gdp_per_capita_usd: float
    population: int


def try_to_extract(label, text):
    print(f"========== {label} ==========")
    print(text)
    print()
    try:
        profile = json.loads(text)
        print("   json.loads SUCCEEDED")
        print("   country   :", profile["country"])
        print("   GDP/capita ($) : ", profile["gdp_per_capita_usd"])
        print("   population  :", profile["population"])
    except json.JSONDecodeError as error:
        print("   json.loads FAILED -> not valid JSON:", error)
    except KeyError as error:
        print("   valid JSON, but a field we need is missing:", error)
    print()


# ---- CASE 1: an ordinary request. No schema. ----
plain = client.models.generate_content(
    model=MODEL,
    contents=QUESTION,
    config=types.GenerateContentConfig(
        system_instruction="Answer the question concisely with no reasoning",
    )
)
try_to_extract("CASE 1: ordinary reply", plain.text)


# ---- CASE 2: same question, but we declare the shape we want. ----
structured = client.models.generate_content(
    model=MODEL,
    contents=QUESTION,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=CountryProfile,          # ...and it must match this form
        system_instruction="Answer the question concisely with no reasoning",
    )
)
try_to_extract("CASE 2: response_schema reply", structured.text)

## 2.2 The System Instruction: Where the Real Control Lives

Every request we send has two separate channels. The **prompt** is the question the user asked. The **system instruction** is the standing orders the model follows no matter what the user asks.

Below, the prompt is byte-for-byte identical in all six calls — `"Tell me about
{city}."` The *only* thing that changes is the standing orders. Watch how far apart the answers land.



In [ ]:
# Same prompt every time; only the system instruction (the model's standing orders) changes.
cities = ["Damascus", "Barcelona"]

system_instructions = {
    "no instruction at all":
        None,
    "geography desk":
        "Answer with only the population, latitude, and longitude. No other text.",
    "news reporter":
        "You are a news reporter. Answer about the latest political climate in one sentence.",
}

for label, instruction in system_instructions.items():
    print(f"########## system_instruction: {label} ##########")
    for city in cities:
        answer = ask_gemini(prompt=f"Tell me about {city}.", system_instruction=instruction, max_output_tokens=50)
        print(f"{city:12s}: {answer}")
    print()

## Follow up Questions and Exercises — Part 2: parameter tweaking

Take a look at the possible parameters you could play with [here](https://ai.google.dev/api/generate-content#generationconfig). Then complete the exercises below.

**Exercise 1.** Experiment with the length of the generated responses (change `max_output_tokens`). What do you observe?

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 2.** Stop the generation at the first period (use `stop_sequences`). What do you observe?

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

# Part 3: Open Weights — The Same Job, Two Very Different Bills

Everything you have done so far in this notebook was **rented**. You never saw Gemini's weights, you cannot download them, and if Google changes the price or retires the model tomorrow, your app changes with it. That is one of exactly two ways to put a language model into a product.

The other way is **open weights**: the company publishes the actual trained parameters, and you run them on hardware you control. Meta's Llama family is the best-known example.

This part sends the same request down both paths.

| | Path A — you host | Path B — you rent |
| --- | --- | --- |
| Model | Llama 3.2 3B Instruct | Llama 4 Scout (17B active, 109B total) |
| Runs on | this Colab's GPU | someone else's datacenter |
| You pay for | the machine, by the hour | the tokens, by the million |
| Works offline | yes, once downloaded | no |
| Ceiling | whatever fits in your GPU | whatever they choose to host |

That last row is the lesson, and you are about to feel it: **the model you can host is not the model you can rent.** A free Colab GPU has about 15 GB of memory. Llama 4 Scout's weights are many times that. It is not a question of patience — it will not fit, at any speed.

## 3.1 Path A: The Weights You Host

Two housekeeping notes before the code.

**Turn on the GPU.** *Runtime → Change runtime type → T4 GPU.* On a CPU runtime the cells below still work, but generation takes minutes instead of seconds.

**About the model name.** Meta's own repository, `meta-llama/Llama-3.2-3B-Instruct`, is *gated*: you must submit your legal name and date of birth to Meta and wait for approval before you can download anything. We use [`unsloth/Llama-3.2-3B-Instruct`](https://huggingface.co/unsloth/Llama-3.2-3B-Instruct) instead — a public mirror of the same weights — so that nobody in this class is blocked by an approval queue. Meta's license still applies either way. Worth noticing on its own: "open weights" and "you can download it right now without asking anyone" are not the same claim.

In [ ]:
# Install the HuggingFace libraries and report which GPU, if any, Colab gave us.
!pip install -q -U transformers accelerate huggingface_hub

import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu} with {total_gb:.1f} GB of memory")
else:
    print("No GPU attached. Runtime -> Change runtime type -> T4 GPU, then re-run this cell.")

In [ ]:
# Download Llama 3.2 3B Instruct onto this machine and load it into the GPU's memory.
from transformers import AutoModelForCausalLM, AutoTokenizer

LOCAL_MODEL = "unsloth/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL,
    dtype=torch.float16,   # the T4 is too old for bfloat16; float16 halves the memory
    device_map="auto",
)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters now sitting on this machine: {n_params / 1e9:.2f} billion")
if torch.cuda.is_available():
    print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

### The step the API was hiding from you

You now own a pile of numbers that predicts the next token. It does not know what a "conversation" is, what a "user" is, or that it should stop talking when your turn comes. Every instruction-tuned model was trained with the conversation flattened into a single string using a specific set of markers, and if you get those markers wrong the model answers as though you had said something else entirely.

`apply_chat_template` writes that string for you, using the format recorded in the model's own files. Print it once. It is the only look you will get at what an API is sending on your behalf every time you call it.

In [ ]:
# Turn a chat-style message list into the exact string Llama was trained to answer.
messages = [
    {"role": "system",
     "content": "You are a customer support triage assistant. Reply with JSON only."},
    {"role": "user",
     "content": ("Ticket: 'I have been charged twice for my March invoice and nobody has "
                 "replied to my last two emails. I have been a customer for six years.' "
                 "Return JSON with the keys: issue, severity (1-5), recommended_action.")},
]

print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

In [ ]:
# Run that prompt on the local model and time how long the generation takes.
import time

torch.manual_seed(BUID)

inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
).to(model.device)

start = time.time()
output = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
local_seconds = time.time() - start

local_answer = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print(local_answer)
print(f"\n--- {local_seconds:.1f} seconds on hardware you controlled, $0.00 in API charges ---")

## 3.2 Path B: The Weights You Rent

Same company, same family, a model roughly thirty times larger — and you will not download a single byte of it.

**Setup, once:**

1. Create a free account at [huggingface.co/join](https://huggingface.co/join).
2. Create a **read** token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens), with *Make calls to Inference Providers* enabled.
3. Save it as a Colab secret named **`MyHFToken`** (the 🔑 icon in the left sidebar → *Add new secret* → *Notebook access* on). As always: never paste a token into a code cell.

A free HuggingFace account comes with a **$0.10 monthly inference credit**. To put that in proportion: one run of the cell below sends about 110 tokens and gets back at most 200, which at Llama 4 Scout's current rates costs on the order of **$0.0002** — two hundredths of a cent. Your credit covers several hundred runs, so re-run it as often as you like. What it will not survive is a loop, so do not write one.

Llama 4 Scout is a **gated** repository, so if Meta has not approved you the call will be refused. The cell below catches that and falls back to an ungated model of comparable size, and the comparison still works. Note what just happened, though: on the hosted path the gate is enforced at *call* time by a third party. That is a dependency you do not have on Path A.

In [ ]:
# Send the identical prompt to a much larger hosted model, falling back if Llama 4 is gated for you.
from huggingface_hub import InferenceClient

API_MODEL = "meta-llama/Llama-4-Scout-17B-16E-Instruct"
FALLBACK_MODEL = "openai/gpt-oss-120b"   # ungated; used automatically if the line above is refused

client = InferenceClient(api_key=get_key('MyHFToken'))


def ask_hosted(model_name, messages, max_tokens=200):
    start = time.time()
    completion = client.chat.completions.create(
        model=model_name,
        messages=messages,
        max_tokens=max_tokens,
        temperature=0.7,
        seed=BUID,
    )
    return completion, time.time() - start


try:
    completion, api_seconds = ask_hosted(API_MODEL, messages)
except Exception as error:
    print(f"{API_MODEL} refused the call:\n  {error}\n\nFalling back to {FALLBACK_MODEL}.\n")
    API_MODEL = FALLBACK_MODEL
    completion, api_seconds = ask_hosted(API_MODEL, messages)

api_answer = completion.choices[0].message.content
print(api_answer)
print(f"\n--- {api_seconds:.1f} seconds, and {completion.usage.prompt_tokens} in + "
      f"{completion.usage.completion_tokens} out = {completion.usage.total_tokens} tokens billed ---")

## 3.3 Side by Side

Byte-identical prompt, byte-identical system instruction. The only thing that changed is where the weights live.

In [ ]:
# Print the two answers together so the difference is something you observe, not something you are told.
print(f"===== {LOCAL_MODEL}  ({local_seconds:.1f}s, no API charge) =====")
print(local_answer.strip())
print()
print(f"===== {API_MODEL}  ({api_seconds:.1f}s, {completion.usage.total_tokens} tokens billed) =====")
print(api_answer.strip())

## Follow up Questions and Exercises — Part 3: open weights

**Question 3.** Both models received a byte-identical prompt. Compare the two answers on one specific thing: could you hand each of them straight to `json.loads`, the way you did in Part 2.1? Report what you actually got, not what you expected. *(5 points)*

**Answer**

*Provide your answer here*

**Question 4.** Suppose your team ships this triage feature and it handles 50,000 tickets a month. Look up the current price per million tokens for the hosted model you actually used — it is on that model's HuggingFace page, and providers change it often — and use the token counts printed above to estimate the monthly API bill. Then say which path you would deploy and why. Cost does not have to be your deciding factor. *(5 points)*

**Answer**

*Provide your answer here*

**Exercise 3.** The 3B model is not doomed to produce unparseable output. Re-run the local generation with a system instruction that shows it one example of exactly the JSON you want and forbids any other text. Does the gap close? (This is the same lever as `response_schema` in Part 2.1 — a constraint on the output, not a bigger model.)

In [ ]:
# Your code here

**Answer**

*Provide your answer here*